# Statistical Significance Tests - Dialect Bias in Housing Scenarios

## Imports and Data Loading

In [ ]:
import json
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings('ignore', category=FutureWarning)

PROJECT_ROOT = Path('.').resolve().parent
OUTPUTS = PROJECT_ROOT / 'outputs'
LEXICON_DIR = PROJECT_ROOT / 'lexicons'

Q_COLS = ['q_AAVE', 'q_NSE', 'q_NP']
LP_COLS = ['log_prob_SAE', 'log_prob_AAVE', 'log_prob_NSE', 'log_prob_NP']
DIALECTS = ['SAE', 'AAVE', 'NSE', 'NP']

DOMAINS = {
    'housing_tenant':   'Tenant (Landlord)',
    'housing_neighbor': 'Neighbour (Community)',
    'housing_roommate': 'Roommate (Personal)',
}
DOMAIN_ORDER = ['housing_tenant', 'housing_neighbor', 'housing_roommate']

MODEL_SHORT = {
    'meta-llama_Llama-3.2-3B':               'Llama-3.2-3B',
    'google_gemma-2-2b':                      'Gemma-2-2B',
    'mistralai_Mistral-7B-v0.3':              'Mistral-7B',
    'mistralai_Mistral-7B-Instruct-v0.3':     'Mistral-7B-Inst',
    'tiiuae_Falcon3-7B-Base':                 'Falcon3-7B',
    'Qwen_Qwen2.5-7B':                       'Qwen2.5-7B',
    'deepseek-ai_DeepSeek-V2-Lite':           'DeepSeek-V2-L',
    '01-ai_Yi-6B':                            'Yi-6B',
    'CohereForAI_aya-expanse-8b':             'Aya-Expanse-8B',
    'microsoft_Phi-3-mini-4k-instruct':       'Phi-3-mini-Inst',
}

def short_name(m):
    return MODEL_SHORT.get(m, m)

frames = []
model_dirs = sorted([d for d in OUTPUTS.iterdir() if d.is_dir() and d.name != 'exploration'])

for d in model_dirs:
    for domain_key in DOMAINS:
        fpath = d / f'{domain_key}_raw_scores.csv'
        if fpath.exists():
            df = pd.read_csv(fpath)
            df['model'] = d.name
            df['domain'] = domain_key
            df['model_short'] = short_name(d.name)
            frames.append(df)

master = pd.concat(frames, ignore_index=True)
master = master.dropna(subset=Q_COLS, how='all')

print(f'Loaded {len(master):,} rows')
print(f'Models: {sorted(master["model_short"].unique())}')
print(f'Domains: {sorted(master["domain"].unique())}')

## Helper: Dunn's Post-Hoc Test

In [ ]:
def dunns_test(groups, group_labels, alpha=0.05):
    """
    Dunn's post-hoc test after Kruskal-Wallis.

    Parameters
    ----------
    groups : list of array-like
        Data for each group.
    group_labels : list of str
        Names for each group.
    alpha : float
        Significance level before Bonferroni correction.

    Returns
    -------
    pd.DataFrame with columns: group_a, group_b, z_stat, p_raw, p_bonf, significant
    """

    all_data = np.concatenate(groups)
    all_ranks = stats.rankdata(all_data)

    N = len(all_data)
    k = len(groups)

    idx = 0
    group_ranks = []
    for g in groups:
        n_g = len(g)
        group_ranks.append(all_ranks[idx:idx + n_g])
        idx += n_g

    mean_ranks = [r.mean() for r in group_ranks]
    group_sizes = [len(g) for g in groups]

    n_comparisons = k * (k - 1) // 2

    _, tie_counts = np.unique(all_ranks, return_counts=True)
    tie_correction = 1 - np.sum(tie_counts**3 - tie_counts) / (N**3 - N)

    results = []
    for (i, label_a), (j, label_b) in combinations(enumerate(group_labels), 2):
        diff = mean_ranks[i] - mean_ranks[j]
        se = np.sqrt(
            tie_correction * (N * (N + 1) / 12) * (1/group_sizes[i] + 1/group_sizes[j])
        )
        if se == 0:
            z = 0.0
            p_raw = 1.0
        else:
            z = diff / se
            p_raw = 2 * stats.norm.sf(abs(z))

        p_bonf = min(p_raw * n_comparisons, 1.0)
        results.append({
            'group_a': label_a,
            'group_b': label_b,
            'mean_rank_a': mean_ranks[i],
            'mean_rank_b': mean_ranks[j],
            'z_stat': z,
            'p_raw': p_raw,
            'p_bonferroni': p_bonf,
            'significant': p_bonf < alpha,
        })

    return pd.DataFrame(results)

print('Dunn\'s test function defined.')

## Primary Test: Kruskal-Wallis H Test Across Four Dialects

In [ ]:
print('=' * 80)
print('KRUSKAL-WALLIS H TEST: Do log-probability distributions differ across dialects?')
print('=' * 80)

kw_results = []

test_configs = [
    ('housing_tenant',   'Tenant (Landlord)'),
    ('housing_neighbor', 'Neighbour (Community)'),
    ('housing_roommate', 'Roommate (Personal)'),
    ('ALL_HOUSING',      'All Housing Combined'),
]

for domain_key, domain_label in test_configs:
    if domain_key == 'ALL_HOUSING':
        subset = master[master['domain'].isin(DOMAIN_ORDER)]
    else:
        subset = master[master['domain'] == domain_key]

    print(f'\n{"=" * 80}')
    print(f'{domain_label}')
    print(f'{"=" * 80}')

    for val in ['negative', 'positive']:
        sv = subset[subset['valence'] == val]

        groups = [sv[lp].dropna().values for lp in LP_COLS]
        group_sizes = [len(g) for g in groups]

        print(f'\n  {val.upper()} adjectives (n per dialect: {dict(zip(DIALECTS, group_sizes))})')

        H_stat, p_val = stats.kruskal(*groups)
        sig = 'SIGNIFICANT' if p_val < 0.05 else 'NOT significant'

        N_total = sum(group_sizes)
        k = len(groups)
        epsilon_sq = (H_stat - k + 1) / (N_total - k) if N_total > k else 0

        print(f'  Kruskal-Wallis H = {H_stat:.2f}, p = {p_val:.2e}, ε² = {epsilon_sq:.4f} → {sig}')

        kw_results.append({
            'scenario': domain_label,
            'valence': val,
            'H_stat': H_stat,
            'p_value': p_val,
            'epsilon_sq': epsilon_sq,
            'significant': p_val < 0.05,
            'N': N_total,
        })

        if p_val < 0.05:
            dunn = dunns_test(groups, DIALECTS)
            print(f'\n  Dunn\'s post-hoc (Bonferroni-corrected):')
            for _, row in dunn.iterrows():
                sig_mark = '***' if row['p_bonferroni'] < 0.001 else '**' if row['p_bonferroni'] < 0.01 else '*' if row['p_bonferroni'] < 0.05 else 'ns'
                print(f'    {row["group_a"]:4s} vs {row["group_b"]:4s}: '
                      f'z = {row["z_stat"]:+7.2f}, '
                      f'p(bonf) = {row["p_bonferroni"]:.2e} {sig_mark}')

print(f'\n\n{"=" * 80}')
print('SUMMARY: Kruskal-Wallis Results')
print(f'{"=" * 80}')
kw_df = pd.DataFrame(kw_results)
print(kw_df.to_string(index=False))

## Pairwise Mann-Whitney U Tests: Each Dialect vs SAE

In [ ]:
print('=' * 80)
print('MANN-WHITNEY U TESTS: Each dialect vs SAE')
print('=' * 80)

mw_results = []

for domain_key, domain_label in test_configs:
    if domain_key == 'ALL_HOUSING':
        subset = master[master['domain'].isin(DOMAIN_ORDER)]
    else:
        subset = master[master['domain'] == domain_key]

    print(f'\n--- {domain_label} ---')

    for val in ['negative', 'positive']:
        sv = subset[subset['valence'] == val]
        sae_vals = sv['log_prob_SAE'].dropna().values

        print(f'  {val} adjectives:')

        for lp_col, dialect in zip(LP_COLS[1:], ['AAVE', 'NSE', 'NP']):
            dialect_vals = sv[lp_col].dropna().values

            U_stat, p_val = stats.mannwhitneyu(sae_vals, dialect_vals, alternative='two-sided')

            n1, n2 = len(sae_vals), len(dialect_vals)
            r_rb = 1 - (2 * U_stat) / (n1 * n2)

            abs_r = abs(r_rb)
            if abs_r < 0.1:
                eff = 'negligible'
            elif abs_r < 0.3:
                eff = 'small'
            elif abs_r < 0.5:
                eff = 'medium'
            else:
                eff = 'large'

            sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'

            if np.mean(dialect_vals) > np.mean(sae_vals):
                direction = f'{dialect} > SAE'
            else:
                direction = f'{dialect} < SAE'

            print(f'    SAE vs {dialect:4s}: U = {U_stat:.0f}, p = {p_val:.2e} {sig}, '
                  f'r = {r_rb:+.4f} ({eff}), {direction}')

            mw_results.append({
                'scenario': domain_label,
                'valence': val,
                'comparison': f'SAE vs {dialect}',
                'U_stat': U_stat,
                'p_value': p_val,
                'rank_biserial_r': r_rb,
                'effect_size': eff,
                'significant': p_val < 0.05,
            })

print(f'\n\n{"=" * 80}')
print('SUMMARY: Mann-Whitney U Results')
print(f'{"=" * 80}')
mw_df = pd.DataFrame(mw_results)
print(mw_df.to_string(index=False))

## Supplementary: One-Sample t-Tests on q-Scores

In [ ]:
print('=' * 80)
print('ONE-SAMPLE t-TESTS: Is mean q-score significantly different from zero?')
print('=' * 80)

ttest_results = []

for domain_key, domain_label in test_configs:
    if domain_key == 'ALL_HOUSING':
        subset = master[master['domain'].isin(DOMAIN_ORDER)]
    else:
        subset = master[master['domain'] == domain_key]

    print(f'\n--- {domain_label} ---')

    for val in ['negative', 'positive']:
        sv = subset[subset['valence'] == val]
        print(f'  {val} adjectives:')

        for qcol, dialect in zip(Q_COLS, ['AAVE', 'NSE', 'NP']):
            q_vals = sv[qcol].dropna()
            n = len(q_vals)
            mean_q = q_vals.mean()
            std_q = q_vals.std()

            t_stat, p_val = stats.ttest_1samp(q_vals, 0)

            d = mean_q / std_q if std_q > 0 else 0
            abs_d = abs(d)
            if abs_d < 0.2:
                eff = 'negligible'
            elif abs_d < 0.5:
                eff = 'small'
            elif abs_d < 0.8:
                eff = 'medium'
            else:
                eff = 'large'

            sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'

            print(f'    q_{dialect:4s}: mean = {mean_q:+.4f}, sd = {std_q:.4f}, '
                  f't({n-1}) = {t_stat:+.2f}, p = {p_val:.2e} {sig}, d = {d:+.3f} ({eff})')

            ttest_results.append({
                'scenario': domain_label,
                'valence': val,
                'dialect': dialect,
                'n': n,
                'mean_q': mean_q,
                'sd_q': std_q,
                't_stat': t_stat,
                'p_value': p_val,
                'cohens_d': d,
                'effect_size': eff,
                'significant': p_val < 0.05,
            })

print(f'\n\n{"=" * 80}')
print('SUMMARY: One-Sample t-Test Results')
print(f'{"=" * 80}')
tt_df = pd.DataFrame(ttest_results)
print(tt_df[['scenario', 'valence', 'dialect', 'mean_q', 'sd_q', 't_stat', 'p_value', 'cohens_d', 'effect_size', 'significant']].to_string(index=False))

## Summary Tables

In [ ]:
print('TABLE 1: Kruskal-Wallis H Test — Dialect Differences in Log-Probability')
print('=' * 90)
print(f'{"Scenario":<30s} {"Valence":<10s} {"H":>10s} {"p":>12s} {"ε²":>8s} {"Sig.":>6s}')
print('-' * 90)
for _, r in kw_df.iterrows():
    p_str = f'{r["p_value"]:.2e}'
    sig = '***' if r['p_value'] < 0.001 else '**' if r['p_value'] < 0.01 else '*' if r['p_value'] < 0.05 else 'ns'
    print(f'{r["scenario"]:<30s} {r["valence"]:<10s} {r["H_stat"]:>10.2f} {p_str:>12s} {r["epsilon_sq"]:>8.4f} {sig:>6s}')

print(f'\n\nTABLE 2: Dunn\'s Post-Hoc (Bonferroni) — All Housing Combined')
print('=' * 90)

for val in ['negative', 'positive']:
    sv = master[master['domain'].isin(DOMAIN_ORDER) & (master['valence'] == val)]
    groups = [sv[lp].dropna().values for lp in LP_COLS]
    dunn = dunns_test(groups, DIALECTS)

    print(f'\n  {val.upper()} adjectives:')
    print(f'  {"Pair":<12s} {"z":>8s} {"p(raw)":>12s} {"p(bonf)":>12s} {"Sig.":>6s}')
    print(f'  {"-"*60}')
    for _, r in dunn.iterrows():
        sig = '***' if r['p_bonferroni'] < 0.001 else '**' if r['p_bonferroni'] < 0.01 else '*' if r['p_bonferroni'] < 0.05 else 'ns'
        pair = f'{r["group_a"]}-{r["group_b"]}'
        print(f'  {pair:<12s} {r["z_stat"]:>+8.2f} {r["p_raw"]:>12.2e} {r["p_bonferroni"]:>12.2e} {sig:>6s}')

print(f'\n\nTABLE 3: One-Sample t-Tests — q-Score ≠ 0 (All Housing Combined)')
print('=' * 90)
combined_tt = [r for r in ttest_results if r['scenario'] == 'All Housing Combined']
print(f'{"Valence":<10s} {"Dialect":<8s} {"Mean q":>10s} {"SD":>8s} {"t":>10s} {"p":>12s} {"d":>8s} {"Sig.":>6s}')
print('-' * 90)
for r in combined_tt:
    sig = '***' if r['p_value'] < 0.001 else '**' if r['p_value'] < 0.01 else '*' if r['p_value'] < 0.05 else 'ns'
    print(f'{r["valence"]:<10s} {r["dialect"]:<8s} {r["mean_q"]:>+10.4f} {r["sd_q"]:>8.4f} {r["t_stat"]:>+10.2f} {r["p_value"]:>12.2e} {r["cohens_d"]:>+8.3f} {sig:>6s}')

print('\n\nSignificance codes: *** p < 0.001, ** p < 0.01, * p < 0.05, ns = not significant')
print('Effect sizes: ε² (Kruskal-Wallis), rank-biserial r (Mann-Whitney), Cohen\'s d (t-test)')